# 09. Carga de documentos diarios en MongoDB

## Objetivo

Validar el proceso completo de generación e inserción de un documento diario en MongoDB Atlas.

En esta primera prueba se procesa una única fecha antes de ampliar el procedimiento a varios días y, posteriormente, al conjunto completo.

In [2]:
from datetime import datetime, timezone
import pandas as pd

from src.database.connection import get_database_engine
from src.database.load_daily_measurements import load_daily_measurements

from src.mongodb.connection import get_mongodb_database
from src.mongodb.daily_summary import build_daily_document
from src.mongodb.daily_plots import (
    generate_quality_weather_plot,
    generate_solar_curves_plot,
)
from src.mongodb.load_documents import (
    create_daily_documents_index,
    upsert_daily_document,
)

In [3]:
test_date = "2023-07-15"
dataset_version = "v3"
collection_name = "daily_summaries"

output_directory = (
    project_root
    / "outputs"
    / "daily_plots"
)

In [4]:
postgres_engine = get_database_engine()

mongodb_database, mongodb_client = get_mongodb_database()

daily_collection = mongodb_database[collection_name]

print("Conexiones establecidas correctamente.")
print(f"Base de datos MongoDB: {mongodb_database.name}")
print(f"Colección: {daily_collection.name}")

Conexiones establecidas correctamente.
Base de datos MongoDB: solar_irradiance_db
Colección: daily_summaries


In [5]:
index_name = create_daily_documents_index(
    daily_collection
)

print(f"Índice disponible: {index_name}")

Índice disponible: uq_daily_document_date_dataset_version


In [6]:
df_day = load_daily_measurements(
    engine=postgres_engine,
    date=test_date,
)

print(f"Fecha procesada: {test_date}")
print(f"Registros recuperados: {len(df_day):,}")

Fecha procesada: 2023-07-15
Registros recuperados: 1,440


In [7]:
daily_document = build_daily_document(
    df=df_day,
    dataset_version=dataset_version,
)

print("Documento diario construido correctamente.")

Documento diario construido correctamente.


In [8]:
df_day = df_day.copy()

df_day["fecha"] = pd.to_datetime(
    df_day["fecha"],
    errors="coerce",
)

if df_day["fecha"].isna().any():
    raise ValueError(
        "Existen fechas que no se han podido convertir correctamente."
    )

df_day["hora_local"] = df_day["fecha"]

In [9]:
solar_curves_path = generate_solar_curves_plot(
    df=df_day,
    output_directory=output_directory,
)

quality_weather_path = generate_quality_weather_plot(
    df=df_day,
    output_directory=output_directory,
)

In [10]:
generation_time = datetime.now(timezone.utc)

daily_document["graficas"] = {
    "curvas_solares": {
        "disponible": True,
        "ruta": solar_curves_path.as_posix(),
        "formato": "png",
        "fecha_generacion": generation_time,
    },
    "calidad_meteorologia": {
        "disponible": True,
        "ruta": quality_weather_path.as_posix(),
        "formato": "png",
        "fecha_generacion": generation_time,
    },
}

daily_document["metadatos"]["updated_at"] = generation_time

In [11]:
result = upsert_daily_document(
    collection=daily_collection,
    document=daily_document,
)

print(f"Documento insertado: {result.upserted_id is not None}")
print(f"Documentos modificados: {result.modified_count}")
print(f"ID insertado: {result.upserted_id}")

Documento insertado: False
Documentos modificados: 1
ID insertado: None


In [12]:
stored_document = daily_collection.find_one(
    {
        "fecha": daily_document["fecha"],
        "dataset.version": dataset_version,
    }
)

if stored_document is None:
    raise ValueError(
        "El documento no se ha encontrado en MongoDB."
    )

print("Documento recuperado correctamente desde MongoDB.")
print(f"MongoDB _id: {stored_document['_id']}")
print(f"Fecha: {stored_document['fecha']}")
print(f"Versión: {stored_document['dataset']['version']}")

Documento recuperado correctamente desde MongoDB.
MongoDB _id: 6a68e332b791d0daa5e1d94a
Fecha: 2023-07-15 00:00:00
Versión: v3


In [13]:
second_result = upsert_daily_document(
    collection=daily_collection,
    document=daily_document,
)

document_count = daily_collection.count_documents(
    {
        "fecha": daily_document["fecha"],
        "dataset.version": dataset_version,
    }
)

print(
    f"Documento insertado en el segundo upsert: "
    f"{second_result.upserted_id is not None}"
)
print(
    f"Documentos modificados en el segundo upsert: "
    f"{second_result.modified_count}"
)
print(
    f"Documentos existentes para la fecha y versión: "
    f"{document_count}"
)

assert second_result.upserted_id is None
assert document_count == 1

print(
    "El segundo upsert no ha generado documentos duplicados."
)

Documento insertado en el segundo upsert: False
Documentos modificados en el segundo upsert: 0
Documentos existentes para la fecha y versión: 1
El segundo upsert no ha generado documentos duplicados.


In [14]:
mongodb_client.close()

print("Conexión con MongoDB cerrada correctamente.")

Conexión con MongoDB cerrada correctamente.


## Resultado

Se ha validado la persistencia de un documento diario en MongoDB Atlas para una fecha de prueba.

El proceso ha permitido:

- establecer las conexiones con PostgreSQL y MongoDB;
- crear el índice único basado en la fecha y la versión del dataset;
- recuperar las mediciones correspondientes al día seleccionado;
- construir el documento de resumen diario;
- generar las gráficas y añadir sus rutas y metadatos;
- insertar el documento mediante una operación `upsert`;
- recuperar posteriormente el documento desde MongoDB;
- repetir el `upsert` sin generar documentos duplicados.

La combinación de los campos `fecha` y `dataset.version` permite identificar de manera unívoca cada documento diario. De esta forma, una nueva ejecución actualiza el documento existente en lugar de crear una copia adicional.

En esta fase se ha procesado únicamente una fecha de prueba. La carga de todas las fechas disponibles se realizará posteriormente mediante un proceso masivo con control individual de errores.